In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive




#### Step 2: Copy the images from Drive

python



In [2]:
import os
import shutil
import glob

DRIVE_DIR = "/content/drive/MyDrive/Processed_SP_ML"
LOCAL_DIR = "/content/Processed_SP_ML"
RESULTS_DIR = "/content/drive/MyDrive/CNN_results"

if not os.path.exists(LOCAL_DIR):
    shutil.copytree(DRIVE_DIR, LOCAL_DIR)

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Images:", len(os.listdir(os.path.join(LOCAL_DIR, "images"))))

Images: 1032




#### Step 3: Libraries

python



In [3]:
import json, time, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, recall_score,
                             roc_curve, hamming_loss, accuracy_score)
from sklearn.ensemble import RandomForestClassifier
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NONE - turn on T4 GPU!")

device: cuda | GPU: Tesla T4




#### Step 4: Settings and the two groups

python



In [4]:
BACKBONE = "densenet121"
PRETRAINED = True
IMG_W, IMG_H = 384, 288
EPOCHS, WARMUP_EPOCHS, PATIENCE = 30, 2, 7
BATCH_SIZE = 16
LR_HEAD, LR_BACKBONE, WEIGHT_DECAY = 1e-3, 1e-4, 1e-4
SEED = 42
LABEL_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]
FEATURES = ["ECD_CD_cells_per_mm2", "ECD_Deficit", "SD", "CV", "HEX_percent", "Donor_Age"]
FINAL_DIR = os.path.join(RESULTS_DIR, f"final_{BACKBONE}")          # Model A results (notebook 06a)
RUN_DIR   = os.path.join(RESULTS_DIR, "defence_original_only")       # Model B + all checks
os.makedirs(RUN_DIR, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

man = pd.read_csv(os.path.join(LOCAL_DIR, "Image_Manifest_MultiLabel.csv"))
man["Label_Origin"] = np.where(man["Tissue_ID"].str.contains("NEB-2023"), "Model-assisted (2023)", "Original")
train_df = man[man["Split"] == "train"].reset_index(drop=True)
test_df  = man[man["Split"] == "test"].reset_index(drop=True)
train_O  = train_df[train_df["Label_Origin"] == "Original"].reset_index(drop=True)   # Model B training data
test_O   = test_df[test_df["Label_Origin"] == "Original"].reset_index(drop=True)     # shared clinician test set
N_FOLDS = int(train_df["CV_Fold"].max()) + 1
A_pred = pd.read_csv(os.path.join(FINAL_DIR, "test_predictions_all.csv"))
print(f"All train {len(train_df)} | Original train {len(train_O)} | Original test {len(test_O)}")
pd.DataFrame({"Train (all)": train_df[LABEL_COLS].sum(), "Train (Original only)": train_O[LABEL_COLS].sum(),
              "Test (Original only)": test_O[LABEL_COLS].sum()})

All train 824 | Original train 462 | Original test 114


,Train (all),Train (Original only),Test (Original only)
FECD,101,35,8
Guttata,81,42,12
Folds,306,163,39
PolyPleo,105,39,9
LowCD,102,36,6




#### Step 5: Dataset and models (same as 06a)

python



In [5]:
IMNET_MEAN, IMNET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
T = torchvision.transforms
TF_TRAIN = T.Compose([T.ToPILImage(),
                      T.RandomResizedCrop((IMG_H, IMG_W), scale=(0.7, 1.0), ratio=(1.25, 1.42)),
                      T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
                      T.ColorJitter(brightness=0.2, contrast=0.2),
                      T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])
TF_EVAL = T.Compose([T.ToPILImage(), T.Resize((IMG_H, IMG_W)),
                     T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])

def fit_tab_scaler(frame):
    """Median-impute + standardise the 6 numbers using the TRAINING rows only."""
    imp = SimpleImputer(strategy="median").fit(frame[FEATURES])
    sc = StandardScaler().fit(imp.transform(frame[FEATURES]))
    return lambda f: sc.transform(imp.transform(f[FEATURES])).astype(np.float32)

class SPDataset(Dataset):
    def __init__(self, frame, tab, train):
        self.paths = [os.path.join(LOCAL_DIR, "images", f) for f in frame["Proc_File"]]
        self.Y = frame[LABEL_COLS].values.astype(np.float32)
        self.tab = torch.tensor(tab, dtype=torch.float32)
        self.tf = TF_TRAIN if train else TF_EVAL
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(cv2.imread(self.paths[i], cv2.IMREAD_GRAYSCALE)), self.tab[i], torch.tensor(self.Y[i])

def make_backbone(name, pretrained):
    W = "DEFAULT" if pretrained else None
    if name == "densenet121":
        m = models.densenet121(weights=W); dim = m.classifier.in_features; m.classifier = nn.Identity(); cam = m.features
    elif name in ("efficientnet_b0", "efficientnet_b2"):
        m = getattr(models, name)(weights=W); dim = m.classifier[1].in_features; m.classifier = nn.Identity(); cam = m.features[-1]
    elif name == "resnet50":
        m = models.resnet50(weights=W); dim = m.fc.in_features; m.fc = nn.Identity(); cam = m.layer4
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=W); dim = m.classifier[2].in_features; m.classifier[2] = nn.Identity(); cam = m.features[-1]
    else:
        raise ValueError(name)
    return m, dim, cam

class ImageOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone, dim, self.cam_layer = make_backbone(BACKBONE, PRETRAINED)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim, len(LABEL_COLS)))
    def forward(self, x, tab=None):
        return self.head(self.backbone(x))

class Fusion(nn.Module):
    """CNN image features + small network on the 6 numbers -> joined -> 5 disease outputs."""
    def __init__(self):
        super().__init__()
        self.backbone, dim, self.cam_layer = make_backbone(BACKBONE, PRETRAINED)
        self.tab_mlp = nn.Sequential(nn.Linear(len(FEATURES), 32), nn.ReLU(), nn.Dropout(0.2),
                                     nn.Linear(32, 32), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim + 32, 128), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(128, len(LABEL_COLS)))
    def forward(self, x, tab):
        return self.head(torch.cat([self.backbone(x), self.tab_mlp(tab)], 1))

print("OK - image-only and fusion models defined")

OK - image-only and fusion models defined




#### Step 6: Training functions (same as 06a)

python



In [6]:
def pos_weight(Y):
    pos = Y.sum(0); neg = len(Y) - pos
    return torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 10), dtype=torch.float32, device=DEVICE)

def make_loader(frame, scale, train):
    return DataLoader(SPDataset(frame, scale(frame), train), batch_size=BATCH_SIZE if train else 32,
                      shuffle=train, drop_last=train, num_workers=2, pin_memory=True)

@torch.no_grad()
def predict(model, dl):
    model.eval(); out = []
    for x, tab, _ in dl:
        x, tab = x.to(DEVICE), tab.to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            p = torch.sigmoid(model(x, tab)) + torch.sigmoid(model(torch.flip(x, dims=[3]), tab))
        out.append((p / 2).float().cpu().numpy())
    return np.concatenate(out)

def macro_auc(Y, P):
    return np.mean([roc_auc_score(Y[:, j], P[:, j]) for j in range(Y.shape[1]) if 0 < Y[:, j].sum() < len(Y)])

def train_one(ModelCls, tr_frame, va_frame=None, epochs=EPOCHS, log=True):
    set_seed()
    scale = fit_tab_scaler(tr_frame)
    tr_dl = make_loader(tr_frame, scale, True)
    va_dl = None if va_frame is None else make_loader(va_frame, scale, False)
    model = ModelCls().to(DEVICE)
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight(tr_frame[LABEL_COLS].values))
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": head_params, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - WARMUP_EPOCHS, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_auc, best_ep, wait = None, -1, epochs, 0
    for ep in range(1, epochs + 1):
        warm = ep <= WARMUP_EPOCHS
        for p in model.backbone.parameters(): p.requires_grad = not warm
        model.train(); t0, losses = time.time(), []
        for x, tab, y in tr_dl:
            x, tab, y = x.to(DEVICE), tab.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                loss = crit(model(x, tab), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
        if not warm: sched.step()
        msg = f"  ep {ep:2d} loss {np.mean(losses):.4f}"
        if va_dl is not None:
            a = macro_auc(va_frame[LABEL_COLS].values, predict(model, va_dl)); msg += f" | val macro-AUC {a:.3f}"
            if a > best_auc: best_auc, best_ep, best, wait = a, ep, copy.deepcopy(model.state_dict()), 0
            elif not warm: wait += 1
        if log: print(msg + (" (warm-up)" if warm else "") + f" ({time.time() - t0:.0f}s)")
        if va_dl is not None and wait >= PATIENCE: break
    if best is not None: model.load_state_dict(best)
    for p in model.parameters(): p.requires_grad = True
    return model, scale, best_ep

print("OK - training functions ready")

OK - training functions ready




#### Step 7: CHECK 1, leakage

python



In [7]:
d_tr, d_te = set(train_df["Donor_Group"]), set(test_df["Donor_Group"])
checks = pd.DataFrame([
    {"Check": "Donors in both train and test", "Result": len(d_tr & d_te)},
    {"Check": "Corneas (Tissue_ID) in both train and test", "Result": len(set(train_df["Tissue_ID"]) & set(test_df["Tissue_ID"]))},
    {"Check": "Image files in both train and test", "Result": len(set(train_df["Proc_File"]) & set(test_df["Proc_File"]))},
    {"Check": "Donors with BOTH eyes in the dataset", "Result": int((man.groupby("Donor_Group").size() > 1).sum())},
    {"Check": "...of which split across train/test", "Result": int((man.groupby("Donor_Group")["Split"].nunique() > 1).sum())},
])
print("CHECK 1 - Leakage (all should be 0, except the 'both eyes' count)")
display(checks)

CHECK 1 - Leakage (all should be 0, except the 'both eyes' count)


,Check,Result
0,Donors in both train and test,0
1,Corneas (Tissue_ID) in both train and test,0
2,Image files in both train and test,0
3,Donors with BOTH eyes in the dataset,233
4,...of which split across train/test,0




#### Step 8: CHECK 2, shuffled labels

python



In [8]:
rf = lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample",
                                    random_state=SEED, n_jobs=-1)
scale = fit_tab_scaler(train_df); Xtr, Xte = scale(train_df), scale(test_df)
rng = np.random.default_rng(SEED); rows = []
for c in LABEL_COLS:
    ytr, yte = train_df[c].values, test_df[c].values
    real = roc_auc_score(yte, rf().fit(Xtr, ytr).predict_proba(Xte)[:, 1])
    shuf = [roc_auc_score(yte, rf().fit(Xtr, rng.permutation(ytr)).predict_proba(Xte)[:, 1]) for _ in range(20)]
    rows.append({"Disease": c, "Real labels AUC": real, "Shuffled labels AUC (mean of 20)": np.mean(shuf),
                 "Shuffled max": np.max(shuf)})
print("CHECK 2 - Shuffled-label test (shuffled should be ~0.5)")
shuffle_tab = pd.DataFrame(rows).set_index("Disease"); display(shuffle_tab.round(3))

CHECK 2 - Shuffled-label test (shuffled should be ~0.5)


,Real labels AUC,Shuffled labels AUC (mean of 20),Shuffled max
Disease,,,
FECD,0.989,0.484,0.769
Guttata,0.592,0.490,0.663
Folds,0.653,0.508,0.572
PolyPleo,0.994,0.469,0.613
LowCD,0.976,0.531,0.793




#### Step 9: CHECK 3, image only on clinician-labelled corneas

python



In [9]:
a = A_pred.merge(test_df[["Tissue_ID", "Label_Origin"]], on="Tissue_ID")
a = a[a["Label_Origin"] == "Original"]
img_tab = pd.DataFrame({"Numbers only": {c: roc_auc_score(a[c], a[f"P_Num_{c}"]) for c in LABEL_COLS},
                        "Image only (no numbers)": {c: roc_auc_score(a[c], a[f"P_Img_{c}"]) for c in LABEL_COLS},
                        "Image + numbers (Model A)": {c: roc_auc_score(a[c], a[f"P_Fus_{c}"]) for c in LABEL_COLS}})
print(f"CHECK 3 - AUC on {len(a)} clinician-labelled test corneas")
display(img_tab.round(3))

CHECK 3 - AUC on 114 clinician-labelled test corneas


,Numbers only,Image only (no numbers),Image + numbers (Model A)
FECD,0.982,0.906,0.985
Guttata,0.619,0.846,0.852
Folds,0.618,0.802,0.824
PolyPleo,0.982,0.876,0.983
LowCD,0.868,0.952,0.955




#### Step 10: CHECK 4, train Model B on clinician labels only (5 folds, saved after each fold)

python



In [10]:
oof_path, info_path = os.path.join(RUN_DIR, "oof_predictions_B.csv"), os.path.join(RUN_DIR, "fold_info_B.json")
oof = pd.read_csv(oof_path) if os.path.exists(oof_path) else \
      train_O[["Tissue_ID", "CV_Fold"] + LABEL_COLS].assign(**{f"P_{l}": np.nan for l in LABEL_COLS})
folds_B = json.load(open(info_path)) if os.path.exists(info_path) else {}
for k in range(N_FOLDS):
    if str(k) in folds_B:
        print(f"Fold {k} already done - skipping"); continue
    tr_f, va_f = train_O[train_O["CV_Fold"] != k], train_O[train_O["CV_Fold"] == k]
    print(f"\n===== Model B | fold {k} | train {len(tr_f)} / val {len(va_f)} =====")
    t0 = time.time()
    model, sc_, best_ep = train_one(Fusion, tr_f, va_f)
    P = predict(model, make_loader(va_f, sc_, False))
    oof.loc[oof["Tissue_ID"].isin(va_f["Tissue_ID"]), [f"P_{l}" for l in LABEL_COLS]] = P
    oof.to_csv(oof_path, index=False)
    folds_B[str(k)] = {"best_epoch": int(best_ep), "val_macro_auc": float(macro_auc(va_f[LABEL_COLS].values, P)),
                       "minutes": round((time.time() - t0) / 60, 1)}
    json.dump(folds_B, open(info_path, "w"), indent=2)
    print(f"Fold {k}: val macro-AUC {folds_B[str(k)]['val_macro_auc']:.3f} - SAVED to Drive")
    del model; torch.cuda.empty_cache()


===== Model B | fold 0 | train 369 / val 93 =====
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 153MB/s]


  ep  1 loss 1.1580 | val macro-AUC 0.671 (warm-up) (8s)
  ep  2 loss 1.1216 | val macro-AUC 0.742 (warm-up) (4s)
  ep  3 loss 1.0627 | val macro-AUC 0.734 (6s)
  ep  4 loss 0.8976 | val macro-AUC 0.825 (6s)
  ep  5 loss 0.8409 | val macro-AUC 0.847 (6s)
  ep  6 loss 0.6716 | val macro-AUC 0.859 (6s)
  ep  7 loss 0.6034 | val macro-AUC 0.871 (6s)
  ep  8 loss 0.5311 | val macro-AUC 0.906 (6s)
  ep  9 loss 0.5662 | val macro-AUC 0.899 (6s)
  ep 10 loss 0.4703 | val macro-AUC 0.882 (6s)
  ep 11 loss 0.5108 | val macro-AUC 0.865 (6s)
  ep 12 loss 0.4521 | val macro-AUC 0.881 (6s)
  ep 13 loss 0.4145 | val macro-AUC 0.874 (6s)
  ep 14 loss 0.3903 | val macro-AUC 0.878 (6s)
  ep 15 loss 0.3643 | val macro-AUC 0.877 (6s)
Fold 0: val macro-AUC 0.906 - SAVED to Drive

===== Model B | fold 1 | train 367 / val 95 =====
  ep  1 loss 1.2072 | val macro-AUC 0.637 (warm-up) (4s)
  ep  2 loss 1.1708 | val macro-AUC 0.788 (warm-up) (4s)
  ep  3 loss 1.1182 | val macro-AUC 0.832 (6s)
  ep  4 loss 1.040



#### Step 11: Final Model B, then predict on the clinician-labelled test corneas

python



In [11]:
ep = max(int(np.median([v["best_epoch"] for v in folds_B.values()])), WARMUP_EPOCHS + 1)
print(f"Final Model B: {ep} epochs on {len(train_O)} clinician-labelled train images")
model_B, scale_B, _ = train_one(Fusion, train_O, None, epochs=ep)
P_B = predict(model_B, make_loader(test_O, scale_B, False))
torch.save(model_B.state_dict(), os.path.join(RUN_DIR, "Model_B_original_only.pt"))
B = test_O[["Tissue_ID"] + LABEL_COLS].copy()
for j, c in enumerate(LABEL_COLS): B[f"P_B_{c}"] = P_B[:, j]
B = B.merge(A_pred[["Tissue_ID"] + [f"P_Fus_{c}" for c in LABEL_COLS]], on="Tissue_ID")
B.to_csv(os.path.join(RUN_DIR, "test_predictions_A_vs_B.csv"), index=False)
print("OK - Model B test predictions ready for", len(B), "clinician-labelled test corneas")

Final Model B: 13 epochs on 462 clinician-labelled train images
  ep  1 loss 1.1811 (warm-up) (4s)
  ep  2 loss 1.1352 (warm-up) (4s)
  ep  3 loss 1.1152 (6s)
  ep  4 loss 1.0046 (6s)
  ep  5 loss 0.8834 (6s)
  ep  6 loss 0.7120 (6s)
  ep  7 loss 0.6827 (6s)
  ep  8 loss 0.5984 (6s)
  ep  9 loss 0.5724 (6s)
  ep 10 loss 0.5601 (6s)
  ep 11 loss 0.5582 (6s)
  ep 12 loss 0.5163 (6s)
  ep 13 loss 0.4992 (6s)
OK - Model B test predictions ready for 114 clinician-labelled test corneas




#### Step 12: Model A vs Model B, with a confidence interval for the difference

python



In [12]:
rng = np.random.default_rng(SEED); rows = []
for c in LABEL_COLS:
    y, pa, pb = B[c].values, B[f"P_Fus_{c}"].values, B[f"P_B_{c}"].values
    diffs = []
    for _ in range(2000):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i): diffs.append(roc_auc_score(y[i], pa[i]) - roc_auc_score(y[i], pb[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    rows.append({"Disease": c, "Cases": int(y.sum()), "Model A (all data)": roc_auc_score(y, pa),
                 "Model B (clinician only)": roc_auc_score(y, pb), "A - B": roc_auc_score(y, pa) - roc_auc_score(y, pb),
                 "95% CI of A - B": f"[{lo:+.3f}, {hi:+.3f}]",
                 "Verdict": "A better" if lo > 0 else ("B better" if hi < 0 else "No real difference")})
ab = pd.DataFrame(rows).set_index("Disease")
cv_B = {c: roc_auc_score(oof[c], oof[f"P_{c}"]) for c in LABEL_COLS}
print("CHECK 4 - Model A vs Model B on the same clinician-labelled test corneas")
display(ab.round(3))
print("Model B cross-validation AUC (clinician-labelled train):", {k: round(v, 3) for k, v in cv_B.items()})

CHECK 4 - Model A vs Model B on the same clinician-labelled test corneas


,Cases,Model A (all data),Model B (clinician only),A - B,95% CI of A - B,Verdict
Disease,,,,,,
FECD,8,0.985,0.971,0.014,"[-0.006, +0.047]",No real difference
Guttata,12,0.852,0.819,0.033,"[-0.006, +0.087]",No real difference
Folds,39,0.824,0.854,-0.031,"[-0.084, +0.022]",No real difference
PolyPleo,9,0.983,0.971,0.012,"[-0.007, +0.038]",No real difference
LowCD,6,0.955,0.921,0.034,"[+0.000, +0.080]",No real difference


Model B cross-validation AUC (clinician-labelled train): {'FECD': np.float64(0.939), 'Guttata': np.float64(0.828), 'Folds': np.float64(0.817), 'PolyPleo': np.float64(0.949), 'LowCD': np.float64(0.96)}




#### Step 13: Save all the evidence

python



In [13]:
checks.to_csv(os.path.join(RUN_DIR, "check1_leakage.csv"), index=False)
shuffle_tab.to_csv(os.path.join(RUN_DIR, "check2_shuffled_labels.csv"))
img_tab.to_csv(os.path.join(RUN_DIR, "check3_image_only_original.csv"))
ab.to_csv(os.path.join(RUN_DIR, "check4_A_vs_B.csv"))
json.dump({"cv_auc_B": cv_B, "epochs_B": ep, "folds_B": folds_B}, open(os.path.join(RUN_DIR, "summary_defence.json"), "w"), indent=2)
print("Saved to", RUN_DIR); print(os.listdir(RUN_DIR))

Saved to /content/drive/MyDrive/CNN_results/defence_original_only
['oof_predictions_B.csv', 'fold_info_B.json', 'Model_B_original_only.pt', 'test_predictions_A_vs_B.csv', 'check1_leakage.csv', 'check2_shuffled_labels.csv', 'check3_image_only_original.csv', 'check4_A_vs_B.csv', 'summary_defence.json']
